# Setup

In [141]:
import os
import numpy as np
import torch
import torch.nn as nn
from transformers.utils import logging

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Suppress warnings
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
logging.set_verbosity_error()
logging.disable_progress_bar()

# 1. Load

In [142]:
from transformers import AutoTokenizer, AutoModel
from torchinfo import summary

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device).eval()

print(model)

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=Tru

In [143]:
from torchinfo import summary

x = tokenizer(
    "I love natural language processing.",
    return_tensors="pt"
)

x = {k: v.to(device) for k, v in x.items()}

model_summary = summary(
    model,
    input_data=x,
    depth=4,
    col_names=["input_size", "output_size", "num_params", "trainable"],
    verbose=0
)

with open("distilbert_summary.txt", "w", encoding="utf-8") as f:
    f.write(str(model_summary))

# 2. Embedding

- Word embedding
  - (V, d) lookup table, where V = vocab size and d = embedding dimension.
- Position embedding
  - Another (T, d) lookup table, where T = maximum number of tokens of the model.
- Normalization
  - Each token $x = [x_1, ..., x_{768}]$ normalizes its 768 features.
  - Typical standardization: $x_i' = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$
    - $\mu$ and $\sigma$ are calculated for each token individually, not accross batch or sequence.
  - Learned scale and shift: $y_i = \gamma_i x_i' + \beta_i$

In [144]:
from transformers import AutoModel

model_name = 'distilbert-base-uncased'
model = AutoModel.from_pretrained(model_name)

embeddings = model.embeddings

## 2.1. Word Embedding

In [145]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokens = tokenizer('I love you')['input_ids']

word_emb_layer = embeddings.word_embeddings
word_emb_matrix = next(word_emb_layer.parameters())
word_emb = word_emb_matrix[tokens]

print(word_emb_matrix.shape)    # (V, d), V=vocab_size, d=embedding dimension
print(word_emb.shape)           # (L, d), L=n_tokens

torch.Size([30522, 768])
torch.Size([5, 768])


## 2.2. Position Embedding

### Learned PE

In [146]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokens = tokenizer('I love you')['input_ids']

position_ids = embeddings.position_ids[0][:len(tokens)]     # [0, 1, 2, 3, 4]
position_emb_matrix = next(embeddings.position_embeddings.parameters())
position_emb = position_emb_matrix[position_ids]

print(position_emb_matrix.shape)    # (T, d), T=max_tokens
print(position_emb.shape)           # (L, d)

torch.Size([512, 768])
torch.Size([5, 768])


### RoPE

- Rotate Q and K.
$$
\begin{bmatrix}
x'_1 \\
x'_2
\end{bmatrix}
=
\begin{bmatrix}
\cos(p\theta) & -\sin(p\theta) \\
\sin(p\theta) & \cos(p\theta)
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2
\end{bmatrix}
$$

In [147]:
# tbd

# 3. Normalization

- $\operatorname{LayerNorm}(x) =
\gamma \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$
- $\operatorname{RMSNorm}(x) =
\gamma \frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d}x_i^2+\epsilon}}$
  - Commonly paired with pre-norm architectures in modern LLMs.
  - Slightly less computation, and empirically strong enough.
- Pre-Norm
  - LayerNorm -> Attention
  - Dominant for modern transformers.
- Post-Norm
  - Attention -> LayerNorm
  - Original transformers like BERT or DistilBERT.

In [148]:
x = word_emb + position_emb         # (L, d)
x = model.embeddings.LayerNorm(x)   # (L, d)

# torch implementation
ln = nn.LayerNorm(768)
rms = nn.RMSNorm(768)

# 4. Attention

## 4.1. Self-Attention

In [149]:
x = word_emb + position_emb     # embedding: (L, d)

attn_layer = model.transformer.layer[0].attention
print(attn_layer)

wq = attn_layer.q_lin       # (d, d)
wk = attn_layer.k_lin       # (d, d)
wv = attn_layer.v_lin       # (d, d)

q = wq(x)      # query: (L, d), what information each token is looking for.
k = wk(x)      # key: (L, d), what features each token offers for matching with queries.
v = wv(x)      # value: (L, d), what information each token provides when attended to.

d = q.shape[1]
attn_score = q.matmul(k.T) / d**0.5     # (L, L): how much each token is related to others.

attn_weights = attn_score.softmax(dim=-1)   # (L, L)
attn_output = attn_weights @ v              # (L, d)

attn_output.shape

DistilBertSelfAttention(
  (q_lin): Linear(in_features=768, out_features=768, bias=True)
  (k_lin): Linear(in_features=768, out_features=768, bias=True)
  (v_lin): Linear(in_features=768, out_features=768, bias=True)
  (out_lin): Linear(in_features=768, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)


torch.Size([5, 768])

## 4.2. Multi-Head Attention

In [150]:
attn_layer = model.transformer.layer[0].attention

h = model.config.n_heads  # h = 12
d = model.config.dim      # d = 768
d_head = d // h           # d_head = 64

q = attn_layer.q_lin(x)   # (L, 768)
k = attn_layer.k_lin(x)   # (L, 768)
v = attn_layer.v_lin(x)   # (L, 768)

# Split into attention heads
q = q.reshape(-1, h, d_head).transpose(0, 1)  # (12, L, 64)
k = k.reshape(-1, h, d_head).transpose(0, 1)  # (12, L, 64)
v = v.reshape(-1, h, d_head).transpose(0, 1)  # (12, L, 64)

attn_score = q @ k.transpose(-2, -1) / d_head**0.5  # (12, L, L)
attn_weights = attn_score.softmax(dim=-1)            # (12, L, L)
attn_output = attn_weights @ v                        # (12, L, 64)

## 4.3. GQA, Grouped-Query Attention

- Calculates $QK^T$.
- 1~7 Q share 1st K/V, 8~14 Q share 2nd K/V.

In [157]:
model = AutoModel.from_pretrained('Qwen/Qwen2.5-0.5B')

attn_layer = model.layers[0].self_attn

print(attn_layer.q_proj)
print(attn_layer.k_proj)
print(attn_layer.v_proj)

q = attn_layer.q_proj(x)
k = attn_layer.k_proj(x)
v = attn_layer.v_proj(x)

# Head sizes
n_q_heads = 14
n_kv_heads = 2
dim_head = 64

# Reshape
q = q.view(1, 4, n_q_heads, dim_head)
k = k.view(1, 4, n_kv_heads, dim_head)
v = v.view(1, 4, n_kv_heads, dim_head)

# Transpose: (B, L, H, D) -> (B, H, L, D)
q = q.transpose(1, 2)
k = k.transpose(1, 2)
v = v.transpose(1, 2)

print("Q: ", q.shape)
print("K: ", k.shape)
print("V: ", v.shape)

Linear(in_features=896, out_features=896, bias=True)
Linear(in_features=896, out_features=128, bias=True)
Linear(in_features=896, out_features=128, bias=True)


RuntimeError: mat1 and mat2 must have the same dtype, but got Float and BFloat16

# 5. FFN

## 5.1. Dense

In [166]:
model = AutoModel.from_pretrained(model_name)

ffn_layer = model.transformer.layer[0].ffn

print(ffn_layer)

FFN(
  (dropout): Dropout(p=0.1, inplace=False)
  (lin1): Linear(in_features=768, out_features=3072, bias=True)
  (lin2): Linear(in_features=3072, out_features=768, bias=True)
  (activation): GELUActivation()
)


## 5.2. SwiGLU MLP

```markdown
x
├─→ Linear 1 → SiLU → gate
└─→ Linear 2        → values

gate * values
      ↓
   Linear 3
      ↓
    output

- gate_proj = Linear 1 = controls how much of each feature passes through.
- up_proj   = Linear 2 = creates features.
- down_proj = Linear 3 = filtered featrues.
```

# 6. Causal mask

- Before softmax, mask future tokens.
$$
\begin{bmatrix}
0 & -\infty & -\infty \\
0 & 0 & -\infty \\
0 & 0 & 0
\end{bmatrix}
$$

# 7. LM head

- Each token's embedding -> one score per vocabulary token.
- (B, L, 896) -> LM head -> (B, L, 151,936)

# 8. KV cache

```markdown
Step 1:
"I"
K_I, V_I → cache

Step 2:
"love"
compute Q_love, K_love, V_love
attention uses:
Q_love vs [K_I, K_love]

cache becomes:
[K_I, K_love]
[V_I, V_love]
```